# Module 49: Advanced Gradient Checkpointing

Go beyond basic `checkpoint(layer, x)`: non-reentrant mode, selective
save/recompute policies, and activation offload ideas.

**Prerequisites:** Module 16 (Activation Checkpointing).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

class MLPBlock(nn.Module):
    def __init__(self, dim, hidden):
        super().__init__()
        self.fc1 = nn.Linear(dim, hidden)
        self.fc2 = nn.Linear(hidden, dim)
    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))

block = MLPBlock(64, 256)
x = torch.randn(4, 16, 64, requires_grad=True)
y = checkpoint(block, x, use_reentrant=False)
y.sum().backward()
print("non-reentrant OK, grad norm:", float(x.grad.norm()))

In [ ]:
# Compare reentrant vs non-reentrant gradients
x = torch.randn(2, 8, 64, requires_grad=True)
block = MLPBlock(64, 128)

y = checkpoint(block, x, use_reentrant=False)
y.sum().backward()
g_nr = x.grad.clone(); x.grad = None

y = checkpoint(block, x, use_reentrant=True)
y.sum().backward()
g_r = x.grad
print("max|diff|", float((g_nr - g_r).abs().max()))

In [ ]:
# Selective idea: checkpoint only the expensive matmul region
w1 = torch.randn(32, 64, requires_grad=True)
w2 = torch.randn(64, 32, requires_grad=True)
h = torch.randn(8, 32, requires_grad=True)

def expensive(t):
    return (t @ w1) @ w2

out = F.gelu(checkpoint(expensive, h, use_reentrant=False))
out.sum().backward()
print("selective-style grads", float(h.grad.norm()), float(w1.grad.norm()))

In [ ]:
# Offload sketch: pack activation to pinned CPU, unpack later
class OffloadSaveContext:
    def __init__(self):
        self.storage = []
    def pack(self, t):
        cpu = t.detach().to("cpu").pin_memory()
        self.storage.append((cpu, t.device))
        return len(self.storage) - 1
    def unpack(self, idx):
        cpu, device = self.storage[idx]
        return cpu.to(device)

ctx = OffloadSaveContext()
act = torch.randn(32, 64)
idx = ctx.pack(act)
restored = ctx.unpack(idx)
print("roundtrip max diff", float((act - restored).abs().max()))

In [ ]:
# Stack: checkpoint every block (LLM / ViT pattern)
layers = nn.ModuleList([MLPBlock(64, 128) for _ in range(4)])
h = torch.randn(2, 16, 64, requires_grad=True)
for layer in layers:
    h = checkpoint(layer, h, use_reentrant=False)
h.mean().backward()
print("stacked checkpoint complete")

## Next Steps

- Run `selective_checkpoint.py` for the full walkthrough
- Profile memory with Module 26 before/after enabling SAC
- Continue to Module 50 for sparse tensors